# Muon Ablation Demo

This notebook runs LR sweeps for all models and optimizers:

1. Clone the repository
2. Install dependencies with `uv`
3. Log in to Weights & Biases (W&B)
4. Run sweeps for **LSTM / Transformer / NanoGPT** with **AdamW + Muon**
5. Plot train/val loss versus elapsed training time

> **Tip:** Demo uses short runs for quick comparison. For full-scale runs, see [README.md](../README.md).

In [ ]:
import os
os.chdir("/content/")
!rm -rf muon_ablation
!git clone https://github.com/MichaelNotDeveloper/muon_ablation.git
os.chdir("muon_ablation")

In [ ]:
!pip install uv
!uv sync

## Weights & Biases login

1. Open [wandb.ai/authorize](https://wandb.ai/authorize) and copy your API key.
2. In Colab, click the **key** icon in the left sidebar → **Secrets**.
3. Add a secret named `WANDB_API_KEY` with your API key.
4. Run the cell below to authenticate.

In [ ]:
import os
from google.colab import userdata
import wandb
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login(key=os.environ["WANDB_API_KEY"])
# !export WANDB_API_KEY={os.environ["WANDB_API_KEY"]}

In [ ]:
import os

BATCH_SIZE = 40

# Common overrides shared by sweep runs.
COMMON_ARGS = " ".join([
    "trainer.n_epochs=20",
    "trainer.override=true",
    "datasets.val.limit=200",
    "writer.mode=online",
    f"batch_size={BATCH_SIZE}",
    "num_workers=4",
    "trainer.compute_matrix_metrics=true",
    "trainer.compute_topsubspace_metrics=true",
    "trainer.dtype=bfloat16",
])

# Dedicated args for time-vs-loss comparison on NanoGPT.
TIME_COMMON_ARGS = " ".join([
    "trainer.n_epochs=10",
    "trainer.epoch_len=50",
    "trainer.override=true",
    "datasets.val.limit=200",
    "writer.mode=online",
    f"batch_size={BATCH_SIZE}",
    "num_workers=2",
    "trainer.compute_matrix_metrics=false",
    "trainer.compute_topsubspace_metrics=false",
    "trainer.dtype=bfloat16",
])

LRS = [1e-5, 2e-5, 5e-5, 1e-4, 2e-4, 5e-4, 1e-3, 2e-3, 5e-3, 1e-2]

print("COMMON_ARGS:", COMMON_ARGS)
print("TIME_COMMON_ARGS:", TIME_COMMON_ARGS)
print("LRS:", LRS)
print("BATCH_SIZE:", BATCH_SIZE)

## LSTM

### AdamW LR sweep (10 values)

In [ ]:
model_name = "lstm"
optimizer_name = "adamw"
for lr in LRS:
    cmd = (
        f"uv run train.py --config-name=lstm optimizer=adamw "
        f"optimizer.lr={lr} lr_scheduler.max_lr={lr} "
        f"writer.run_name={optimizer_name}_{model_name}_{lr}_b_{BATCH_SIZE} {COMMON_ARGS}"
    )
    print("Running:", cmd)
    os.system(cmd)

### Muon LR sweep (10 values)

In [ ]:
# Muon LR sweep (10 values)
model_name = "lstm"
optimizer_name = "muon"
for lr in LRS:
    cmd = (
        f"uv run train.py --config-name=lstm optimizer=muon "
        f"optimizer.muon.lr={lr} optimizer.adam.lr={lr/10} lr_scheduler.max_lr={lr} "
        f"writer.run_name={optimizer_name}_{model_name}_{lr}_b_{BATCH_SIZE} {COMMON_ARGS}"
    )
    print("Running:", cmd)
    os.system(cmd)

## Transformer

### AdamW LR sweep (10 values)

In [ ]:
model_name = "transformer"
optimizer_name = "adamw"
for lr in LRS:
    cmd = (
        f"uv run train.py --config-name=transformer optimizer=adamw "
        f"optimizer.lr={lr} lr_scheduler.max_lr={lr} "
        f"writer.run_name={optimizer_name}_{model_name}_{lr}_b_{BATCH_SIZE} {COMMON_ARGS}"
    )
    print("Running:", cmd)
    os.system(cmd)

### Muon LR sweep (10 values)

In [ ]:
# Muon LR sweep (10 values)
model_name = "transformer"
optimizer_name = "muon"
for lr in LRS:
    cmd = (
        f"uv run train.py --config-name=transformer optimizer=muon "
        f"optimizer.muon.lr={lr} optimizer.adam.lr={lr/10} lr_scheduler.max_lr={lr} "
        f"writer.run_name={optimizer_name}_{model_name}_{lr}_b_{BATCH_SIZE} {COMMON_ARGS}"
    )
    print("Running:", cmd)
    os.system(cmd)

## NanoGPT

### AdamW LR sweep (10 values)
### Muon LR sweep (10 values)

In [ ]:
model_name = "nanogpt"
optimizer_name = "adamw"
for lr in LRS:
    cmd = (
        f"uv run train.py --config-name=nanogpt optimizer=adamw "
        f"optimizer.lr={lr} lr_scheduler.max_lr={lr} "
        f"writer.run_name={optimizer_name}_{model_name}_{lr}_b_{BATCH_SIZE} {COMMON_ARGS}"
    )
    print("Running:", cmd)
    os.system(cmd)

In [ ]:
model_name = "nanogpt"
optimizer_name = "muon"
for lr in LRS:
    cmd = (
        f"uv run train.py --config-name=nanogpt optimizer=muon "
        f"optimizer.muon.lr={lr} optimizer.adam.lr={lr/10} lr_scheduler.max_lr={lr} "
        f"writer.run_name={optimizer_name}_{model_name}_{lr}_b_{BATCH_SIZE} {COMMON_ARGS}"
    )
    print("Running:", cmd)
    os.system(cmd)

## NanoGPT Time Comparison

Run two time-focused experiments (AdamW vs Muon), then plot loss over elapsed time.

In [ ]:
cmd = (
    "uv run train.py --config-name=nanogpt optimizer=adamw "
    "writer.run_name=adamw_time "
    f"{TIME_COMMON_ARGS}"
)
print("Running:", cmd)
os.system(cmd)

### Muon Time Run

In [ ]:
cmd = (
    "uv run train.py --config-name=nanogpt optimizer=muon "
    "writer.run_name=muon_time "
    f"{TIME_COMMON_ARGS}"
)
print("Running:", cmd)
os.system(cmd)

## Final NanoGPT Run (Full OpenWebText)

This run switches from `openwebtext_10k` to the full `openwebtext` dataset config.

In [ ]:
cmd = (
    "uv run train.py --config-name=nanogpt datasets=openwebtext optimizer=muon "
    "writer.run_name=muon_nanogpt_full_openwebtext "
    f"{TIME_COMMON_ARGS}"
)
print("Running:", cmd)
os.system(cmd)

In [ ]:
!uv run python -m scripts.plot_loss_vs_time